# Preprocessing - Partidos internacionales de fútbol

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('clean-data', exist_ok=True)
print('Directorio clean-data/ listo.')

Directorio clean-data/ listo.


In [ ]:
results = pd.read_csv('data/results.csv', encoding='utf-8')
goalscorers = pd.read_csv('data/goalscorers.csv', encoding='utf-8')
shootouts = pd.read_csv('data/shootouts.csv', encoding='utf-8')
former_names = pd.read_csv('data/former_names.csv', encoding='utf-8')

print('Shapes:', results.shape, goalscorers.shape, shootouts.shape, former_names.shape)
results.head(3)

Shapes: (49287, 9) (47601, 8) (675, 5) (36, 4)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


In [3]:
# Parsear fechas
results['date'] = pd.to_datetime(results['date'])

# Eliminar partidos futuros que tienen scores NaN
results_clean = results.dropna(subset=['home_score']).copy()
print(f'Eliminados {len(results) - len(results_clean)} partidos futuros sin resultado.')

# Castear scores a int
results_clean['home_score'] = results_clean['home_score'].astype(int)
results_clean['away_score'] = results_clean['away_score'].astype(int)

print(results_clean.dtypes)
results_clean.head(3)

Eliminados 72 partidos futuros sin resultado.
date          datetime64[ns]
home_team             object
away_team             object
home_score             int32
away_score             int32
tournament            object
city                  object
country               object
neutral                 bool
dtype: object


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False


In [4]:
# Resultado del partido
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] == row['away_score']:
        return 'draw'
    else:
        return 'away_win'

results_clean['result'] = results_clean.apply(get_result, axis=1)
results_clean['score_diff'] = results_clean['home_score'] - results_clean['away_score']
results_clean['total_goals'] = results_clean['home_score'] + results_clean['away_score']
results_clean['year'] = results_clean['date'].dt.year
results_clean['decade'] = (results_clean['year'] // 10) * 10

print('Distribución de resultados:')
print(results_clean['result'].value_counts())
results_clean[['date','home_team','away_team','result','total_goals','year','decade']].head(5)

Distribución de resultados:
result
home_win    24106
away_win    13912
draw        11197
Name: count, dtype: int64


,date,home_team,away_team,result,total_goals,year,decade
0,1872-11-30,Scotland,England,draw,0,1872,1870
1,1873-03-08,England,Scotland,home_win,6,1873,1870
2,1874-03-07,Scotland,England,home_win,3,1874,1870
3,1875-03-06,England,Scotland,draw,4,1875,1870
4,1876-03-04,Scotland,England,home_win,3,1876,1870


In [5]:
# Parsear fechas en goalscorers
goalscorers['date'] = pd.to_datetime(goalscorers['date'])

results_keys = set(
    results_clean['date'].astype(str) + '_' +
    results_clean['home_team'] + '_' +
    results_clean['away_team']
)
gs_keys = (
    goalscorers['date'].astype(str) + '_' +
    goalscorers['home_team'] + '_' +
    goalscorers['away_team']
)
goalscorers_clean = goalscorers[gs_keys.isin(results_keys)].copy()
print(f'Goles conservados: {len(goalscorers_clean)} / {len(goalscorers)}')
goalscorers_clean.head(3)

Goles conservados: 47601 / 47601


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False


In [6]:
# Parsear fechas en shootouts y former_names
shootouts['date'] = pd.to_datetime(shootouts['date'])
former_names['start_date'] = pd.to_datetime(former_names['start_date'])
former_names['end_date'] = pd.to_datetime(former_names['end_date'])

print('shootouts:', shootouts.shape)
print('former_names:', former_names.shape)
shootouts.head(3)

shootouts: (675, 5)
former_names: (36, 4)


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN


In [7]:
results_clean.to_csv('clean-data/results_clean.csv', index=False, encoding='utf-8')
goalscorers_clean.to_csv('clean-data/goalscorers_clean.csv', index=False, encoding='utf-8')
shootouts.to_csv('clean-data/shootouts_clean.csv', index=False, encoding='utf-8')

print('Guardados en clean-data/:')
for f in os.listdir('clean-data'):
    path = os.path.join('clean-data', f)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

Guardados en clean-data/:
  goalscorers_clean.csv  (3318.8 KB)
  results_clean.csv  (4745.8 KB)
  shootouts_clean.csv  (28.7 KB)
